## Imports e configurações

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
sns.set_style('whitegrid')

PALETTE = {'primary':'#0B3D91','teal':'#00A6A6','muted':'#6B7280','accent':'#F59E0B'}
ROOT = Path.cwd()
OUTPUT = ROOT / "outputs"
merged = pd.read_csv(OUTPUT / "conciliation_results_with_rates.csv", parse_dates=['data_referencia','data_cessao','data_vencimento'])
merged.head()


FileNotFoundError: [Errno 2] No such file or directory: 'c:\\Users\\Caah\\Documents\\case_facio_financial_reconciliation\\notebooks\\outputs\\conciliation_results_with_rates.csv'

## KPIs e tabelas

In [ ]:
# KPIs
with open(OUTPUT / 'kpis_summary.json') as f:
    import json
    kpis = json.load(f)
kpis


## Exposição por status e produto (Q2)

In [ ]:
# VP por status
vp_status = merged.groupby('recon_status')['valor_presente_calculado'].sum().sort_values(ascending=False)
vp_status.plot(kind='bar', color=[PALETTE['primary'] if s=='Match Exact' else PALETTE['teal'] for s in vp_status.index], figsize=(8,5))
plt.title('VP total por status')
plt.ylabel('Valor Presente (R$)')
plt.show()

# VP por produto x status
exposure = merged.groupby(['produto','recon_status'])['valor_presente_calculado'].sum().unstack(fill_value=0)
exposure.head()


## Distribuição de divergências e top10 (Q2)

In [ ]:
plt.figure(figsize=(8,5))
sns.boxplot(x='recon_status', y='abs_diff', data=merged, palette=[PALETTE['primary'],PALETTE['teal'],PALETTE['muted']])
plt.yscale('symlog')  # se houver outliers extremos
plt.title('Boxplot divergências por status')
plt.ylabel('Diferença absoluta (R$)')
plt.show()

top10 = merged.sort_values('abs_diff', ascending=False).head(10)
top10[['id_contrato','parcela','produto','fundo','valor_presente_calculado','valor_presente_fundo','abs_diff','i_facio','i_fundo','dc_cessao']]


## Análise de taxa implícita (Q3)

In [ ]:
mask = (merged['recon_status']=='Match Divergent') & (~merged['i_fundo'].isna())
merged.loc[mask, 'i_diff'] = merged.loc[mask, 'i_fundo'] - merged.loc[mask, 'i_facio']
plt.figure(figsize=(8,5))
sns.histplot(merged.loc[mask,'i_diff'].dropna(), bins=50, color=PALETTE['teal'])
plt.title('Distribuição da diferença de taxa (i_fundo - i_facio) para divergentes')
plt.xlabel('Diferença de taxa diária (decimal)')
plt.show()
merged.loc[mask, ['produto','fundo']].value_counts()


## Composição e concentração (Q4)

In [ ]:
merged['days_to_maturity'] = (merged['data_vencimento'] - merged['data_referencia']).dt.days
def bucket(x):
    if pd.isna(x): return 'unknown'
    if x < 30: return '<30'
    if 30 <= x <= 90: return '30-90'
    if 90 < x <= 180: return '90-180'
    return '>180'
merged['bucket'] = merged['days_to_maturity'].apply(bucket)
vp_by_bucket = merged.groupby('bucket')['valor_presente_calculado'].sum().reindex(['<30','30-90','90-180','>180','unknown'])
vp_by_bucket.plot(kind='bar', color=PALETTE['primary'], figsize=(8,5))
plt.title('Distribuição do VP por bucket de prazo')
plt.ylabel('Valor Presente (R$)')
plt.show()

# prazo médio ponderado por VP por fundo
pmvp = merged.groupby('fundo').apply(lambda g: (g['valor_presente_calculado'] * g['days_to_maturity']).sum() / g['valor_presente_calculado'].sum())
pmvp = pmvp.rename('prazo_medio_ponderado_dias')
pmvp
